### Analysis of IPL Data Set 2008 to 2026

## win percentage by team

In [20]:
%%writefile analysis.py
import pandas as pd
from collections import defaultdict

df = pd.read_csv("data/processed/IPL_Dataset_Cleaned.csv", low_memory=False)

TEAM_REBRAND_NAME = {
    "Delhi Daredevils": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings",
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru",
    "Rising Pune Supergiants": "Rising Pune Supergiant",
}

def compute_team_win_percentage(df):
    df = df.copy()
    df['match_won_by_clean'] = df['match_won_by'].replace(TEAM_REBRAND_NAME)

    match_level = df.groupby('match_id').agg(
        teams=('batting_team_clean', lambda x: sorted(set(x))),
        won_by=('match_won_by_clean', 'first'),
    ).reset_index()

    decided = match_level[match_level['won_by'] != 'Unknown']

    played, wins = defaultdict(int), defaultdict(int)
    for _, row in decided.iterrows():
        for t in row['teams']:
            played[t] += 1
        wins[row['won_by']] += 1

    team_stats = pd.DataFrame({
        'team': list(played.keys()),
        'matches_played': [played[t] for t in played],
        'wins': [wins.get(t, 0) for t in played],
    })
    team_stats['win_pct'] = (team_stats['wins'] / team_stats['matches_played'] * 100).round(1)
    return team_stats.sort_values('win_pct', ascending=False).reset_index(drop=True)




Overwriting analysis.py


In [21]:
from analysis import compute_team_win_percentage
import pandas as pd
import importlib
import analysis
importlib.reload(analysis)
df = pd.read_csv("data/processed/IPL_Dataset_Cleaned.csv", low_memory=False)
team_stats = compute_team_win_percentage(df)
print(team_stats)

                           team  matches_played  wins  win_pct
0        Rising Pune Supergiant              16    15     93.8
1                Gujarat Titans              77    47     61.0
2           Chennai Super Kings             264   148     56.1
3                Mumbai Indians             287   155     54.0
4         Kolkata Knight Riders             271   140     51.7
5   Royal Challengers Bengaluru             280   143     51.1
6              Rajasthan Royals             245   123     50.2
7           Sunrisers Hyderabad             206   102     49.5
8          Lucknow Super Giants              70    34     48.6
9                  Punjab Kings             271   126     46.5
10               Delhi Capitals             272   125     46.0
11                Gujarat Lions              29    13     44.8
12         Kochi Tuskers Kerala              14     6     42.9
13              Deccan Chargers              75    29     38.7
14                Pune Warriors              45    12  

In [22]:
print(open("analysis.py").read())

import pandas as pd
from collections import defaultdict

df = pd.read_csv("data/processed/IPL_Dataset_Cleaned.csv", low_memory=False)

TEAM_REBRAND_NAME = {
    "Delhi Daredevils": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings",
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru",
    "Rising Pune Supergiants": "Rising Pune Supergiant",
}

def compute_team_win_percentage(df):
    df = df.copy()
    df['match_won_by_clean'] = df['match_won_by'].replace(TEAM_REBRAND_NAME)

    match_level = df.groupby('match_id').agg(
        teams=('batting_team_clean', lambda x: sorted(set(x))),
        won_by=('match_won_by_clean', 'first'),
    ).reset_index()

    decided = match_level[match_level['won_by'] != 'Unknown']

    played, wins = defaultdict(int), defaultdict(int)
    for _, row in decided.iterrows():
        for t in row['teams']:
            played[t] += 1
        wins[row['won_by']] += 1

    team_stats = pd.DataFrame({
        'team': list(played.keys()),
       

### How has each team's performance trended season by season?

In [23]:
%%writefile -a analysis.py

def compute_team_win_percentage_by_season(df):
    """
    Business Question 2: season-by-season win percentage per team.
    Returns a team x season pivot table. A blank (NaN) cell means that
    team did not play that season - not a 0% loss record.
    """

    df = df.copy()
    df['match_won_by_clean'] = df['match_won_by'].replace(TEAM_REBRAND_NAME)

    match_level= df.groupby('match_id').agg(
        season= ('season_clean', 'first'),
        teams=('batting_team_clean', lambda x: sorted(set(x))),
        won_by=('match_won_by_clean', 'first')
    ).reset_index()

    decided = match_level[match_level['won_by'] != 'Unknown']


    played, wins = defaultdict(int), defaultdict(int)
    for _, row in decided.iterrows():
        for t in row['teams']:
            played[(t, row['season'])] += 1
        wins[(row['won_by'], row['season'])] +=1


    rows = [{'team': t, 'season':s, 'win_pct': round(wins.get((t,s),0)/p*100, 1)}
               for (t, s), p in played.items()]

    return pd.DataFrame(rows).pivot(index='team', columns='season', values='win_pct')
    

Appending to analysis.py


In [24]:
import importlib
import analysis
importlib.reload(analysis)
from analysis import compute_team_win_percentage_by_season
season_pivot = compute_team_win_percentage_by_season(df)
season_pivot

season,2008,2009,2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
team,,,,,,,,,,,,,,,,,,,
Chennai Super Kings,56.2,57.1,60.0,68.8,55.6,66.7,62.5,58.8,NaN,NaN,68.8,58.8,42.9,68.8,28.6,66.7,50.0,28.6,42.9
Deccan Chargers,14.3,56.2,50.0,42.9,26.7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Delhi Capitals,50.0,66.7,50.0,30.8,61.1,20.0,14.3,38.5,50.0,42.9,35.7,60.0,50.0,60.0,50.0,35.7,50.0,50.0,50.0
Gujarat Lions,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,56.2,30.8,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Gujarat Titans,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,75.0,64.7,41.7,60.0,58.8
Kochi Tuskers Kerala,NaN,NaN,NaN,42.9,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Kolkata Knight Riders,46.2,25.0,50.0,53.3,70.6,37.5,73.3,53.8,53.3,56.2,56.2,46.2,46.2,52.9,42.9,42.9,78.6,41.7,41.7
Lucknow Super Giants,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,60.0,57.1,50.0,42.9,30.8
Mumbai Indians,50.0,38.5,68.8,62.5,58.8,68.4,46.7,62.5,50.0,68.8,42.9,66.7,78.6,50.0,28.6,56.2,28.6,56.2,28.6


## Head-to-head: which teams dominate which opponents?

In [25]:
%%writefile -a analysis.py
def compute_head_to_head(df):
    """
    Business Question 3: head-to-head win percentage between every pair
    of teams that have played each other at least once.
    Returns a team x team pivot: row_team's win % against column_team.
    Diagonal and never-played pairs are NaN.
    """

    df = df.copy()
    df['match_won_by_clean'] =df['match_won_by'].replace(TEAM_REBRAND_NAME)

    match_level= df.groupby('match_id').agg(
        teams = ('batting_team_clean', lambda x:sorted(set(x))),
        won_by=('match_won_by_clean', 'first')
    ).reset_index()

    decided = match_level[(match_level['won_by'] !='Unknown') & (match_level['teams'].apply(len)==2)]


    h2h_wins = defaultdict(int)
    for _, row in decided.iterrows():
        a,b = row['teams']
        loser = b if row['won_by'] == a else a
        h2h_wins[(row['won_by'], loser)] +=1

    teams = sorted(set(df['batting_team_clean'].unique()))
    rows = []
    for a in teams:
        for b in teams:
            if a == b:
                continue
            wins_a, wins_b = h2h_wins.get((a, b), 0), h2h_wins.get((b, a), 0)
            total = wins_a + wins_b
            if total > 0:
                rows.append({'team': a, 'opponent': b, 'win_pct': round(wins_a / total * 100, 1)})

    return pd.DataFrame(rows).pivot(index='team', columns='opponent', values='win_pct')

Appending to analysis.py


In [26]:
import importlib
import analysis
importlib.reload(analysis)
from analysis import compute_head_to_head

head_to_head = compute_head_to_head(df)
print(head_to_head)

opponent                     Chennai Super Kings  Deccan Chargers  \
team                                                                
Chennai Super Kings                          NaN             60.0   
Deccan Chargers                             40.0              NaN   
Delhi Capitals                              36.4             63.6   
Gujarat Lions                                NaN              NaN   
Gujarat Titans                              60.0              NaN   
Kochi Tuskers Kerala                        50.0              0.0   
Kolkata Knight Riders                       34.4             77.8   
Lucknow Super Giants                        57.1              NaN   
Mumbai Indians                              51.2             60.0   
Pune Warriors                               33.3             25.0   
Punjab Kings                                50.0             70.0   
Rajasthan Royals                            50.0             77.8   
Rising Pune Supergiant            

In [27]:
print(open("analysis.py").read())

import pandas as pd
from collections import defaultdict

df = pd.read_csv("data/processed/IPL_Dataset_Cleaned.csv", low_memory=False)

TEAM_REBRAND_NAME = {
    "Delhi Daredevils": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings",
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru",
    "Rising Pune Supergiants": "Rising Pune Supergiant",
}

def compute_team_win_percentage(df):
    df = df.copy()
    df['match_won_by_clean'] = df['match_won_by'].replace(TEAM_REBRAND_NAME)

    match_level = df.groupby('match_id').agg(
        teams=('batting_team_clean', lambda x: sorted(set(x))),
        won_by=('match_won_by_clean', 'first'),
    ).reset_index()

    decided = match_level[match_level['won_by'] != 'Unknown']

    played, wins = defaultdict(int), defaultdict(int)
    for _, row in decided.iterrows():
        for t in row['teams']:
            played[t] += 1
        wins[row['won_by']] += 1

    team_stats = pd.DataFrame({
        'team': list(played.keys()),
       

In [16]:
%%writefile -a analysis.py
def compute_top_run_scorers(df, top_n=15):
    """
    Business Question 4: all-time top run-scorers.
    Sums 'runs_batter' (runs on that specific ball) - NOT 'batter_runs',
    which is a cumulative running total within the innings and would
    wildly overcount if summed directly.
    """
    runs = df.groupby('batter')['runs_batter'].sum()
    matches = df.groupby('batter')['match_id'].nunique()
    innings = df.groupby('batter').apply(lambda x: x[['match_id', 'innings']].drop_duplicates().shape[0])

    top_scorers = pd.DataFrame({
        'total_runs': runs,
        'matches': matches,
        'innings': innings,
    }).sort_values('total_runs', ascending=False).head(top_n)
    top_scorers['avg_runs_per_innings'] = (top_scorers['total_runs'] / top_scorers['innings']).round(1)
    return top_scorers

Appending to analysis.py


In [18]:

import importlib
import analysis
importlib.reload(analysis)

from analysis import compute_top_run_scorers
top_scores = compute_top_run_scorers(df)
print(top_scores)

E:\neha work\Data Analytics\Projects\IPL sales analaysis\analysis.py:582: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pp_by_innings = df[df['over'] <= 5].groupby(['match_id', 'innings']).apply(phase_rr)
E:\neha work\Data Analytics\Projects\IPL sales analaysis\analysis.py:583: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  death_by_innings = df[df['over'] >= 15].groupby(['match_id', 'innings']).apply(phase_rr)


t=27.01, p=3.3572470322135823e-140, mean diff=2.05 runs/over, n=2356 innings


E:\neha work\Data Analytics\Projects\IPL sales analaysis\analysis.py:609: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pp_by_innings = df[df['over'] <= 5].groupby(['match_id', 'innings']).apply(phase_rr)
E:\neha work\Data Analytics\Projects\IPL sales analaysis\analysis.py:610: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  death_by_innings = df[df['over'] >= 15].groupby(['match_id', 'innings']).apply(phase_rr)


t=27.01, p=3.3572470322135823e-140, mean diff=2.05 runs/over, n=2356 innings
                total_runs  matches  innings  avg_runs_per_innings
batter                                                            
V Kohli               9346      275      277                  33.7
RG Sharma             7331      275      276                  26.6
S Dhawan              6769      221      222                  30.5
DA Warner             6567      184      187                  35.1
KL Rahul              5828      149      152                  38.3
SK Raina              5536      200      201                  27.5
MS Dhoni              5439      241      241                  22.6
AM Rahane             5367      197      197                  27.2
AB de Villiers        5181      170      172                  30.1
SV Samson             5181      185      185                  28.0
CH Gayle              4997      141      145                  34.5
RV Uthappa            4954      197      198        

E:\neha work\Data Analytics\Projects\IPL sales analaysis\analysis.py:690: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.


In [30]:
print(open("analysis.py").read())

import pandas as pd
from collections import defaultdict

df = pd.read_csv("data/processed/IPL_Dataset_Cleaned.csv", low_memory=False)

TEAM_REBRAND_NAME = {
    "Delhi Daredevils": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings",
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru",
    "Rising Pune Supergiants": "Rising Pune Supergiant",
}

def compute_team_win_percentage(df):
    df = df.copy()
    df['match_won_by_clean'] = df['match_won_by'].replace(TEAM_REBRAND_NAME)

    match_level = df.groupby('match_id').agg(
        teams=('batting_team_clean', lambda x: sorted(set(x))),
        won_by=('match_won_by_clean', 'first'),
    ).reset_index()

    decided = match_level[match_level['won_by'] != 'Unknown']

    played, wins = defaultdict(int), defaultdict(int)
    for _, row in decided.iterrows():
        for t in row['teams']:
            played[t] += 1
        wins[row['won_by']] += 1

    team_stats = pd.DataFrame({
        'team': list(played.keys()),
       

## Who are the all-time top wicket-takers?

In [31]:
%%writefile -a analysis.py
import pandas as pd
def compute_top_wicket_takers(df, top_n=15):
    """
    Business Question 5: all-time top wicket-takers.
    Sums 'bowler_wicket' (per-ball flag) - NOT 'team_wicket', which is
    the batting team's cumulative wicket count and isn't bowler-specific
    at all.
    """
    wickets = df.groupby('bowler')['bowler_wicket'].sum()
    matches=df.groupby('bowler')['match_id'].nunique()
    balls_bowled = df[df['valid_ball']==1].groupby('bowler').size()


    top_bowlers = pd.DataFrame({
        'total_wickets':wickets,
        'matches':matches,
        'balls_bowled':balls_bowled,
        
    }).sort_values('total_wickets', ascending=False).head(top_n)
    top_bowlers['wickets_per_match'] =(top_bowlers['total_wickets']/top_bowlers['matches']).round(2)
    return top_bowlers
    


    
  

Appending to analysis.py


In [32]:
import importlib
import analysis
importlib.reload(analysis)
from analysis import compute_top_wicket_takers

top_bowlers = compute_top_wicket_takers(df)
print(top_bowlers)

                 total_wickets  matches  balls_bowled  wickets_per_match
bowler                                                                  
YS Chahal                  233      184          4037               1.27
B Kumar                    226      206          4600               1.10
SP Narine                  209      200          4660               1.04
PP Chawla                  192      191          3850               1.01
JJ Bumrah                  190      158          3653               1.20
R Ashwin                   187      217          4710               0.86
DJ Bravo                   183      158          3120               1.16
RA Jadeja                  180      237          4254               0.76
Rashid Khan                179      153          3543               1.17
A Mishra                   174      162          3371               1.07
SL Malinga                 170      122          2827               1.39
HV Patel                   151      121          25

In [33]:
print(open("analysis.py").read())

import pandas as pd
from collections import defaultdict

df = pd.read_csv("data/processed/IPL_Dataset_Cleaned.csv", low_memory=False)

TEAM_REBRAND_NAME = {
    "Delhi Daredevils": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings",
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru",
    "Rising Pune Supergiants": "Rising Pune Supergiant",
}

def compute_team_win_percentage(df):
    df = df.copy()
    df['match_won_by_clean'] = df['match_won_by'].replace(TEAM_REBRAND_NAME)

    match_level = df.groupby('match_id').agg(
        teams=('batting_team_clean', lambda x: sorted(set(x))),
        won_by=('match_won_by_clean', 'first'),
    ).reset_index()

    decided = match_level[match_level['won_by'] != 'Unknown']

    played, wins = defaultdict(int), defaultdict(int)
    for _, row in decided.iterrows():
        for t in row['teams']:
            played[t] += 1
        wins[row['won_by']] += 1

    team_stats = pd.DataFrame({
        'team': list(played.keys()),
       

### Best batting strike rate (minimum-balls cutoff)

In [34]:
%%writefile -a analysis.py
import pandas as pd

def compute_top_strike_rates(df, min_balls=1500, top_n=15):
    """
    Business Question 6: best batting strike rate, minimum 1500 balls faced
    so a small hot streak doesn't outrank a real season of scoring.
    balls_faced correctly excludes wides and includes no-balls - verified
    against valid_ball and cross-checked against batter_balls.
    """
    runs = df.groupby('batter')['runs_batter'].sum()
    balls = df.groupby('batter')['balls_faced'].sum()
    matches = df.groupby('batter')['match_id'].nunique()

    sr_df = pd.DataFrame({'total_runs': runs, 'balls_faced': balls, 'matches': matches})
    sr_df = sr_df[sr_df['balls_faced'] >= min_balls].copy()
    sr_df['strike_rate'] = (sr_df['total_runs'] / sr_df['balls_faced'] * 100).round(1)
    return sr_df.sort_values('strike_rate', ascending=False).head(top_n)

Appending to analysis.py


In [35]:
import importlib
import analysis
importlib.reload(analysis)
from analysis import compute_top_strike_rates

Strike_rates = compute_top_strike_rates(df)
print(Strike_rates)

                 total_runs  balls_faced  matches  strike_rate
batter                                                        
AD Russell             2655         1525      114        174.1
N Pooran               2527         1543      100        163.8
V Sehwag               2728         1755      104        155.4
GJ Maxwell             2820         1818      134        155.1
YBK Jaiswal            2593         1697       82        152.8
AB de Villiers         5181         3411      170        151.9
JC Buttler             4647         3105      136        149.7
CH Gayle               4997         3346      141        149.3
B Sai Sudharsan        2515         1686       57        149.2
SA Yadav               4581         3084      164        148.5
KA Pollard             3437         2329      168        147.6
MP Stoinis             2242         1520      108        147.5
RR Pant                3879         2644      136        146.7
HH Pandya              2964         2025      150      

### Best bowling economy (minimum-overs cutoff)

In [36]:
%%writefile -a analysis.py

def compute_top_economy_bowlers(df, min_balls=1200, top_n=15):
    """
    Business Question 7: best economy rate, minimum 200 overs (1200 legal
    balls) bowled. runs_bowler correctly excludes byes/leg-byes on legal
    deliveries - verified: runs_bowler == runs_batter on every legal ball
    in the dataset - while still including wide/no-ball penalties.
    """
    legal = df[df['valid_ball'] == 1]
    balls_bowled = legal.groupby('bowler').size()
    runs_conceded = df.groupby('bowler')['runs_bowler'].sum()

    eco_df = pd.DataFrame({'runs_conceded': runs_conceded, 'balls_bowled': balls_bowled}).dropna()
    eco_df = eco_df[eco_df['balls_bowled'] >= min_balls].copy()
    eco_df['overs_bowled'] = (eco_df['balls_bowled'] / 6).round(1)
    eco_df['economy'] = (eco_df['runs_conceded'] / (eco_df['balls_bowled'] / 6)).round(2)
    return eco_df.sort_values('economy').head(top_n)

Appending to analysis.py


In [37]:
import importlib
import analysis
importlib.reload(analysis)
from analysis import compute_top_economy_bowlers

eco_df = compute_top_economy_bowlers(df)
print(eco_df)

                 runs_conceded  balls_bowled  overs_bowled  economy
bowler                                                             
M Muralitharan            1706          1528         254.7     6.70
SP Narine                 5273          4660         776.7     6.79
DW Steyn                  2523          2182         363.7     6.94
Harbhajan Singh           4030          3416         569.3     7.08
SL Malinga                3366          2827         471.2     7.14
R Ashwin                  5652          4710         785.0     7.20
Rashid Khan               4297          3543         590.5     7.28
JJ Bumrah                 4469          3653         608.8     7.34
PP Ojha                   2332          1899         316.5     7.37
A Mishra                  4145          3371         561.8     7.38
R Bhatia                  2020          1636         272.7     7.41
AR Patel                  4437          3585         597.5     7.43
Shakib Al Hasan           1839          1484    

## Which players are the most consistent run-scorers, season to season?

In [38]:
%%writefile -a analysis.py

def compute_batting_consistency(df, min_innings_per_season=5, min_seasons=6, min_career_avg=20, top_n=10):
    """
    Business Question 8: most consistent run-scorers, season to season.
    Consistency = coefficient of variation (std / mean) of each player's
    average runs-per-innings across qualifying seasons. Lower CV = more
    consistent.
    """
    innings_level = df.groupby(['batter', 'season_clean', 'match_id', 'innings'])['runs_batter'].sum().reset_index()
    season_level = innings_level.groupby(['batter', 'season_clean']).agg(
        innings_played=('runs_batter', 'count'),
        avg_runs=('runs_batter', 'mean')
    ).reset_index()

    real_seasons = season_level[season_level['innings_played'] >= min_innings_per_season]
    seasons_per_player = real_seasons.groupby('batter').size()
    qualified_players = seasons_per_player[seasons_per_player >= min_seasons].index

    consistency = real_seasons[real_seasons['batter'].isin(qualified_players)].groupby('batter')['avg_runs'].agg(['mean', 'std', 'count'])
    consistency['cv'] = (consistency['std'] / consistency['mean']).round(3)
    consistency = consistency.rename(columns={'mean': 'career_avg_runs', 'count': 'qualifying_seasons'})

    return consistency[consistency['career_avg_runs'] >= min_career_avg].sort_values('cv').head(top_n)

Appending to analysis.py


In [39]:
import importlib
import analysis
importlib.reload(analysis)
from analysis import compute_batting_consistency

consistency_df = compute_batting_consistency(df)
print(consistency_df)

             career_avg_runs       std  qualifying_seasons     cv
batter                                                           
R Dravid           26.253379  2.713879                   6  0.103
BB McCullum        24.978134  3.733954                   9  0.149
N Rana             25.375860  3.838309                   9  0.151
S Dhawan           30.530737  4.869613                  16  0.159
DR Smith           27.747898  4.574438                   7  0.165
JP Duminy          26.578079  4.404345                   7  0.166
RG Sharma          26.779982  4.479994                  19  0.167
SV Samson          27.811713  5.309871                  14  0.191
SK Raina           27.372385  5.279856                  13  0.193
M Vijay            25.471142  4.983946                   7  0.196


## Does winning the toss correlate with winning the match, and has that changed over time?

In [40]:
%%writefile -a analysis.py

def compute_toss_match_correlation(df):
    """
    Business Question 9: does winning the toss correlate with winning the
    match, and has toss strategy shifted over the 19 seasons?
    """
    df = df.copy()
    df['toss_winner_clean'] = df['toss_winner'].replace(TEAM_REBRAND_NAME)
    df['match_won_by_clean'] = df['match_won_by'].replace(TEAM_REBRAND_NAME)

    match_level = df.groupby('match_id').agg(
        season=('season_clean', 'first'),
        toss_winner=('toss_winner_clean', 'first'),
        toss_decision=('toss_decision', 'first'),
        won_by=('match_won_by_clean', 'first'),
    ).reset_index()

    decided = match_level[match_level['won_by'] != 'Unknown'].copy()
    decided['toss_winner_won_match'] = decided['toss_winner'] == decided['won_by']

    overall_pct = round(decided['toss_winner_won_match'].mean() * 100, 1)
    by_decision = decided.groupby('toss_decision')['toss_winner_won_match'].mean().mul(100).round(1)

    by_season = decided.groupby('season').agg(
        toss_win_match_pct=('toss_winner_won_match', lambda x: round(x.mean() * 100, 1)),
        field_first_pct=('toss_decision', lambda x: round((x == 'field').mean() * 100, 1)),
    )
    return overall_pct, by_decision, by_season

Appending to analysis.py


In [41]:
import importlib
import analysis
importlib.reload(analysis)
from analysis import compute_toss_match_correlation

overall_pct, by_decision, by_season = compute_toss_match_correlation(df)
print("Overall:", overall_pct, "%")
print(by_decision)

Overall: 51.6 %
toss_decision
bat      45.3
field    54.7
Name: toss_winner_won_match, dtype: float64


## Is chasing more successful than defending?

In [42]:
%%writefile -a analysis.py

def compute_chase_vs_defend(df):
    """
    Business Question 10: is chasing more successful than defending,
    overall and by season?
    """
    df = df.copy()
    df['match_won_by_clean'] = df['match_won_by'].replace(TEAM_REBRAND_NAME)

    first_innings_team = df[df['innings'] == 1].groupby('match_id')['batting_team_clean'].first()
    second_innings_team = df[df['innings'] == 2].groupby('match_id')['batting_team_clean'].first()

    match_level = df.groupby('match_id').agg(
        season=('season_clean', 'first'),
        won_by=('match_won_by_clean', 'first'),
    ).reset_index()
    match_level['team_bat_2nd'] = match_level['match_id'].map(second_innings_team)

    decided = match_level.dropna(subset=['team_bat_2nd'])
    decided = decided[decided['won_by'] != 'Unknown'].copy()
    decided['chasing_team_won'] = decided['team_bat_2nd'] == decided['won_by']

    overall_pct = round(decided['chasing_team_won'].mean() * 100, 1)
    by_season = decided.groupby('season')['chasing_team_won'].mean().mul(100).round(1)
    return overall_pct, by_season

Appending to analysis.py


In [43]:
import importlib
import analysis
importlib.reload(analysis)
from analysis import compute_chase_vs_defend

overall_pct, by_season = compute_chase_vs_defend(df)
print("Overall chase win rate:", overall_pct, "%")

Overall chase win rate: 54.3 %


### Which venues produce the highest/lowest average first-innings scores?

In [44]:
%%writefile -a analysis.py

def compute_venue_scoring(df, min_matches=20):
    """
    Business Question 11: average first-innings score by venue.
    Uses the final (max) team_runs of innings 1 as that match's
    first-innings total.
    """
    first_innings = df[df['innings'] == 1].groupby('match_id').agg(
        venue=('venue_clean', 'first'),
        final_score=('team_runs', 'max')
    ).reset_index()

    venue_scores = first_innings.groupby('venue')['final_score'].agg(['mean', 'count']).round(1)
    venue_scores = venue_scores[venue_scores['count'] >= min_matches].sort_values('mean', ascending=False)
    return venue_scores.rename(columns={'mean': 'avg_first_innings_score', 'count': 'matches'})

Appending to analysis.py


In [45]:
import importlib
import analysis
importlib.reload(analysis)
from analysis import compute_venue_scoring

venue_scores = compute_venue_scoring(df)
print(venue_scores)

                                                    avg_first_innings_score  \
venue                                                                         
Narendra Modi Stadium, Ahmedabad                                      180.7   
Brabourne Stadium                                                     178.5   
Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cr...                    174.8   
M Chinnaswamy Stadium                                                 174.1   
Wankhede Stadium                                                      173.1   
Arun Jaitley Stadium                                                  171.8   
Sawai Mansingh Stadium                                                169.1   
Eden Gardens                                                          168.6   
Maharashtra Cricket Association Stadium                               168.3   
Punjab Cricket Association IS Bindra Stadium                          168.3   
Rajiv Gandhi International Stadium                  

## What do the biggest winning margins look like?

In [46]:
%%writefile -a analysis.py

def compute_biggest_margins(df, top_n=10):
    """
    Business Question 12: biggest winning margins, by runs and by wickets
    (wickets uses balls-remaining as a tiebreaker for the 10-wicket ties).
    """
    df = df.copy()
    df['match_won_by_clean'] = df['match_won_by'].replace(TEAM_REBRAND_NAME)

    second_innings_balls = df[df['innings'] == 2].groupby('match_id')['team_balls'].max()

    match_level = df.groupby('match_id').agg(
        season=('season_clean', 'first'), won_by=('match_won_by_clean', 'first'),
        win_margin=('win_margin', 'first'), win_type=('win_type', 'first'),
    ).reset_index()
    match_level['balls_remaining'] = 120 - match_level['match_id'].map(second_innings_balls)

    biggest_runs = match_level[match_level['win_type'] == 'runs'].sort_values('win_margin', ascending=False).head(top_n)
    biggest_wickets = match_level[(match_level['win_type'] == 'wickets') & (match_level['win_margin'] == 10)].sort_values('balls_remaining', ascending=False).head(top_n)
    closest = match_level.dropna(subset=['win_margin']).sort_values('win_margin').head(top_n)

    return biggest_runs, biggest_wickets, closest

Appending to analysis.py


In [47]:
import importlib
import analysis
importlib.reload(analysis)
from analysis import compute_biggest_margins

biggest_runs, biggest_wickets, closest = compute_biggest_margins(df)
print(biggest_runs[['won_by','season','win_margin']])
print(biggest_wickets[['won_by','season','balls_remaining']])
print(closest[['won_by','season','win_margin','win_type']])

                           won_by  season  win_margin
620                Mumbai Indians    2017       146.0
560   Royal Challengers Bengaluru    2016       144.0
0           Kolkata Knight Riders    2008       140.0
497   Royal Challengers Bengaluru    2015       138.0
351   Royal Challengers Bengaluru    2013       130.0
706           Sunrisers Hyderabad    2019       118.0
1009  Royal Challengers Bengaluru    2023       112.0
236                  Punjab Kings    2011       111.0
1161          Sunrisers Hyderabad    2025       110.0
1039        Kolkata Knight Riders    2024       106.0
                           won_by  season  balls_remaining
60                 Delhi Capitals    2009             91.0
611                  Punjab Kings    2017             73.0
683   Royal Challengers Bengaluru    2018             71.0
1080          Sunrisers Hyderabad    2024             62.0
483   Royal Challengers Bengaluru    2015             57.0
124   Royal Challengers Bengaluru    2010           

### Which matches were the closest, and do any teams/venues keep showing up in them?

In [48]:
%%writefile -a analysis.py

def compute_close_matches(df, runs_cutoff=10, wickets_cutoff=1):
    """
    Business Question 13: closest matches, and which teams/venues appear
    most often among them (as a % of their total matches, not raw count).
    """
    match_level = df.groupby('match_id').agg(
        venue=('venue_clean', 'first'), win_margin=('win_margin', 'first'), win_type=('win_type', 'first'),
        teams=('batting_team_clean', lambda x: sorted(set(x))),
    ).reset_index()

    close = match_level[((match_level['win_type'] == 'runs') & (match_level['win_margin'] <= runs_cutoff)) |
                         ((match_level['win_type'] == 'wickets') & (match_level['win_margin'] <= wickets_cutoff))]

    team_close = pd.Series([t for teams in close['teams'] for t in teams]).value_counts()
    team_total = pd.Series([t for teams in match_level.dropna(subset=['win_margin'])['teams'] for t in teams]).value_counts()
    team_stats = pd.DataFrame({'close_matches': team_close, 'total_matches': team_total})
    team_stats['pct_close'] = (team_stats['close_matches'] / team_stats['total_matches'] * 100).round(1)

    venue_close = close['venue'].value_counts()
    return team_stats.sort_values('pct_close', ascending=False), venue_close

Appending to analysis.py


In [49]:
import pandas as pd
df = pd.read_csv("data/processed/IPL_Dataset_Cleaned.csv", low_memory=False)

import importlib
import analysis
importlib.reload(analysis)
from analysis import compute_close_matches

team_stats, venue_close = compute_close_matches(df)
print(team_stats.head(8))
print(venue_close.head(6))

                        close_matches  total_matches  pct_close
Rising Pune Supergiant              3             16       18.8
Lucknow Super Giants               12             70       17.1
Deccan Chargers                    10             75       13.3
Kolkata Knight Riders              35            271       12.9
Punjab Kings                       33            271       12.2
Sunrisers Hyderabad                25            206       12.1
Rajasthan Royals                   28            245       11.4
Mumbai Indians                     31            287       10.8
venue
Eden Gardens                          16
Arun Jaitley Stadium                  15
Wankhede Stadium                      14
MA Chidambaram Stadium                13
Rajiv Gandhi International Stadium    12
M Chinnaswamy Stadium                  9
Name: count, dtype: int64


### How has average scoring changed across the 19 seasons?

In [50]:
%%writefile -a analysis.py

def compute_scoring_trend(df):
    """
    Business Question 14: average first-innings score by season, plus a
    correlation test to check whether the trend is statistically real.
    """
    from scipy import stats

    first_innings = df[df['innings'] == 1].groupby('match_id').agg(
        season=('season_clean', 'first'),
        final_score=('team_runs', 'max')
    ).reset_index()

    by_season = first_innings.groupby('season')['final_score'].mean().round(1)
    corr, pval = stats.pearsonr(first_innings['season'], first_innings['final_score'])
    return by_season, corr, pval

Appending to analysis.py


In [51]:
import importlib
import analysis
importlib.reload(analysis)
from analysis import compute_scoring_trend

by_season, corr, pval = compute_scoring_trend(df)
print(by_season)
print(f"Correlation: r={corr:.3f}, p={pval:.4f}")

season
2008    161.0
2009    150.3
2010    164.8
2011    152.4
2012    157.5
2013    155.9
2014    163.1
2015    166.3
2016    162.6
2017    165.8
2018    172.5
2019    166.7
2020    169.5
2021    159.3
2022    171.1
2023    182.7
2024    189.6
2025    188.8
2026    193.0
Name: final_score, dtype: float64
Correlation: r=0.317, p=0.0000


### Powerplay vs. death-overs scoring

In [52]:
%%writefile -a analysis.py

def compute_powerplay_vs_death(df):
    """
    Business Question 15: powerplay (overs 1-6) vs death-overs (16-20)
    scoring, overall and by season.
    """
    def phase_run_rate(phase_df):
        legal_balls = (phase_df['valid_ball'] == 1).sum()
        return round(phase_df['runs_total'].sum() / (legal_balls / 6), 2)

    pp, death = df[df['over'] <= 5], df[df['over'] >= 15]
    overall = {'powerplay_rr': phase_run_rate(pp), 'death_rr': phase_run_rate(death)}

    rows = []
    for s in sorted(df['season_clean'].unique()):
        s_pp = df[(df['over'] <= 5) & (df['season_clean'] == s)]
        s_death = df[(df['over'] >= 15) & (df['season_clean'] == s)]
        rows.append({'season': s, 'powerplay_rr': phase_run_rate(s_pp), 'death_rr': phase_run_rate(s_death)})
    by_season = pd.DataFrame(rows).set_index('season')

    return overall, by_season

Appending to analysis.py


In [53]:
import importlib
import analysis
importlib.reload(analysis)
from analysis import compute_powerplay_vs_death

overall, by_season = compute_powerplay_vs_death(df)
print(overall)
print(by_season)

{'powerplay_rr': np.float64(8.09), 'death_rr': np.float64(10.06)}
        powerplay_rr  death_rr
season                        
2008            7.70      9.95
2009            7.25      9.13
2010            7.90      9.59
2011            7.20      9.11
2012            7.22      9.52
2013            6.97      9.57
2014            7.56     10.04
2015            7.77     10.31
2016            7.70     10.17
2017            8.34      9.64
2018            8.42     10.14
2019            8.14     10.17
2020            7.72     10.20
2021            7.72      9.41
2022            7.82     10.45
2023            8.72     10.49
2024            9.47     11.21
2025            9.62     10.96
2026           10.10     10.79


In [54]:
import pandas as pd
df = pd.read_csv("data/processed/IPL_Dataset_Cleaned.csv", low_memory=False)

In [55]:
%%writefile -a analysis.py

def compute_batting_average_and_boundaries(df, min_dismissals=30, top_n=15):
    """
    Phase 7 EDA: batting average (runs per dismissal - NOT runs per innings,
    which Question 8 already covered) and boundary counts.
    """
    runs = df.groupby('batter')['runs_batter'].sum()
    dismissals = df.groupby('batter')['striker_out'].sum()
    fours = df[df['runs_batter'] == 4].groupby('batter').size()
    sixes = df[df['runs_batter'] == 6].groupby('batter').size()

    stats_df = pd.DataFrame({'runs': runs, 'dismissals': dismissals, 'fours': fours, 'sixes': sixes}).fillna(0)
    stats_df = stats_df[stats_df['dismissals'] >= min_dismissals].copy()
    stats_df['batting_average'] = (stats_df['runs'] / stats_df['dismissals']).round(1)
    stats_df['boundary_runs'] = stats_df['fours'] * 4 + stats_df['sixes'] * 6
    stats_df['boundary_pct_of_runs'] = (stats_df['boundary_runs'] / stats_df['runs'] * 100).round(1)

    return stats_df.sort_values('batting_average', ascending=False).head(top_n)

Appending to analysis.py


In [56]:
import importlib
import analysis
importlib.reload(analysis)
from analysis import compute_batting_average_and_boundaries

avg_df = compute_batting_average_and_boundaries(df)
print(avg_df)

                 runs  dismissals  fours  sixes  batting_average  \
batter                                                             
B Sai Sudharsan  2515          51  258.0   83.0             49.3   
KL Rahul         5828         125  509.0  239.0             46.6   
H Klaasen        2104          49  146.0  120.0             42.9   
AB de Villiers   5181         122  414.0  253.0             42.5   
JP Duminy        2029          48  126.0   79.0             42.3   
DA Warner        6567         158  663.0  236.0             41.6   
Shubman Gill     4598         111  446.0  152.0             41.4   
V Kohli          9346         229  847.0  317.0             40.8   
JC Buttler       4647         115  459.0  211.0             40.4   
MEK Hussey       1977          49  198.0   52.0             40.3   
SE Marsh         2489          62  269.0   78.0             40.1   
CH Gayle         4997         125  408.0  359.0             40.0   
RD Gaikwad       2839          72  260.0  107.0 


## EDA Item 2 — Team-Level Scoring & Wickets

In [57]:
%%writefile -a analysis.py

def compute_team_scoring_profile(df):
    """
    Phase 7 EDA: team-level average runs scored/conceded and wickets
    lost/taken per innings (not win % - Question 1 already covers that).
    Restricted to regular innings (1-2); Super Overs excluded since a
    1-over mini-innings would badly skew a per-innings average.
    """
    regular = df[df['innings'] <= 2]
    innings_final = regular.groupby(['match_id', 'innings']).agg(
        batting_team=('batting_team_clean', 'first'),
        bowling_team=('bowling_team_clean', 'first'),
        final_runs=('team_runs', 'max'),
        final_wkts=('team_wicket', 'max'),
    ).reset_index()

    batting_side = innings_final.groupby('batting_team').agg(
        avg_runs_scored=('final_runs', 'mean'), avg_wkts_lost=('final_wkts', 'mean')
    ).round(1)
    bowling_side = innings_final.groupby('bowling_team').agg(
        avg_runs_conceded=('final_runs', 'mean'), avg_wkts_taken=('final_wkts', 'mean')
    ).round(1)

    team_profile = batting_side.join(bowling_side)
    team_profile['run_diff'] = (team_profile['avg_runs_scored'] - team_profile['avg_runs_conceded']).round(1)
    return team_profile.sort_values('run_diff', ascending=False)

Appending to analysis.py


In [58]:
import importlib
import analysis
importlib.reload(analysis)
from analysis import compute_team_scoring_profile

team_profile = compute_team_scoring_profile(df)
print(team_profile)

                             avg_runs_scored  avg_wkts_lost  \
batting_team                                                  
Chennai Super Kings                    164.5            5.4   
Gujarat Titans                         178.5            5.4   
Mumbai Indians                         163.6            6.0   
Rising Pune Supergiant                 154.4            5.6   
Sunrisers Hyderabad                    164.8            5.9   
Punjab Kings                           164.4            6.1   
Royal Challengers Bengaluru            161.9            5.8   
Kolkata Knight Riders                  156.9            5.9   
Rajasthan Royals                       161.6            6.0   
Deccan Chargers                        152.8            6.4   
Lucknow Super Giants                   177.1            6.2   
Rising Pune Supergiants                147.4            4.8   
Delhi Capitals                         157.9            6.0   
Kochi Tuskers Kerala                   135.8           

### EDA Item 3 (wrap-up) — Match Records & Season Overview

In [59]:
%%writefile -a analysis.py

def compute_score_records(df, top_n=5):
    """Phase 7 EDA: highest and lowest single-innings scores (completed matches only)."""
    decided_matches = df[df['win_outcome'].notna()]['match_id'].unique()
    regular = df[(df['innings'] <= 2) & (df['match_id'].isin(decided_matches))]

    innings_final = regular.groupby(['match_id', 'innings']).agg(
        team=('batting_team_clean', 'first'), season=('season_clean', 'first'), score=('team_runs', 'max')
    ).reset_index()

    highest = innings_final.sort_values('score', ascending=False).head(top_n)[['team', 'season', 'score']]
    lowest = innings_final.sort_values('score').head(top_n)[['team', 'season', 'score']]
    return highest, lowest


def compute_season_overview(df):
    """Phase 7 EDA: matches played and average wickets per match, by season."""
    matches_by_season = df.groupby('season_clean')['match_id'].nunique()
    wkts_by_season = df[df['innings'] <= 2].groupby(['season_clean', 'match_id', 'innings'])['team_wicket'].max().groupby('season_clean').mean().round(2)
    return pd.DataFrame({'matches': matches_by_season, 'avg_wickets_per_innings': wkts_by_season})

Appending to analysis.py


In [60]:
import importlib
import analysis
importlib.reload(analysis)
from analysis import compute_score_records, compute_season_overview

highest, lowest = compute_score_records(df)
overview_df = compute_season_overview(df)
print(highest)
print(lowest)
print(overview_df)

                       team  season  score
2068    Sunrisers Hyderabad    2024    287
2154    Sunrisers Hyderabad    2025    286
2276    Sunrisers Hyderabad    2025    278
2024    Sunrisers Hyderabad    2024    277
2040  Kolkata Knight Riders    2024    272
                             team  season  score
847           Sunrisers Hyderabad    2014     44
1247        Kolkata Knight Riders    2017     48
1189  Royal Challengers Bengaluru    2017     49
87            Chennai Super Kings    2008     55
119              Rajasthan Royals    2009     58
              matches  avg_wickets_per_innings
season_clean                                  
2008               58                     5.94
2009               57                     6.11
2010               60                     6.00
2011               73                     5.61
2012               74                     5.79
2013               76                     5.98
2014               60                     5.59
2015               59    

## Advanced Analysis

### Team Momentum

In [67]:
%%writefile -a analysis.py

def compute_team_momentum(df):
    """
    Phase 8: does winning the previous match predict winning the next one?
    Returns (baseline_pct, after_win_pct, after_loss_pct, chi2, p_value).
    """
    from scipy.stats import chi2_contingency

    df = df.copy()
    df['match_won_by_clean'] = df['match_won_by'].replace(TEAM_REBRAND_NAME)

    match_level = df.groupby('match_id').agg(
        date=('date', 'first'), won_by=('match_won_by_clean', 'first'),
        teams=('batting_team_clean', lambda x: sorted(set(x))),
    ).reset_index()
    match_level['date'] = pd.to_datetime(match_level['date'], format='%Y-%m-%d')
    decided = match_level[(match_level['won_by'] != 'Unknown') & (match_level['teams'].apply(len) == 2)].sort_values('date')

    rows = [{'team': t, 'date': row['date'], 'won': t == row['won_by']}
            for _, row in decided.iterrows() for t in row['teams']]
    team_form = pd.DataFrame(rows).sort_values(['team', 'date'])
    team_form['prev_result'] = team_form.groupby('team')['won'].shift(1)

    baseline = round(team_form['won'].mean() * 100, 1)
    after_win = round(team_form[team_form['prev_result'] == True]['won'].mean() * 100, 1)
    after_loss = round(team_form[team_form['prev_result'] == False]['won'].mean() * 100, 1)

    valid = team_form.dropna(subset=['prev_result'])
    chi2, p, dof, exp = chi2_contingency(pd.crosstab(valid['prev_result'], valid['won']))
    return baseline, after_win, after_loss, round(chi2, 3), round(p, 4)

Appending to analysis.py


In [68]:
import importlib
import analysis
importlib.reload(analysis)
from analysis import compute_team_momentum

baseline, after_win, after_loss, chi2, p = compute_team_momentum(df)
print(f"Baseline win rate: {baseline}%")
print(f"Win rate after a win: {after_win}%")
print(f"Win rate after a loss: {after_loss}%")
print(f"Chi-square: {chi2}, p-value: {p}")

Baseline win rate: 49.8%
Win rate after a win: 49.2%
Win rate after a loss: 50.4%
Chi-square: 0.324, p-value: 0.569


## Toss Impact & Chase Advantage — Formal Significance Tests

In [65]:
%%writefile -a analysis.py

def compute_toss_chase_significance(df):
    """
    Phase 8: formal significance tests for toss impact and chase advantage.
    """
    from scipy.stats import binomtest, chi2_contingency

    df = df.copy()
    df['toss_winner_clean'] = df['toss_winner'].replace(TEAM_REBRAND_NAME)
    df['match_won_by_clean'] = df['match_won_by'].replace(TEAM_REBRAND_NAME)

    match_level = df.groupby('match_id').agg(
        toss_winner=('toss_winner_clean', 'first'), toss_decision=('toss_decision', 'first'),
        won_by=('match_won_by_clean', 'first'),
    ).reset_index()
    decided = match_level[match_level['won_by'] != 'Unknown'].copy()
    decided['toss_winner_won'] = decided['toss_winner'] == decided['won_by']

    n_toss_wins, n_total = decided['toss_winner_won'].sum(), len(decided)
    test1_p = binomtest(n_toss_wins, n_total, 0.5).pvalue

    chi2, test2_p, _, _ = chi2_contingency(pd.crosstab(decided['toss_decision'], decided['toss_winner_won']))

    second_innings_team = df[df['innings'] == 2].groupby('match_id')['batting_team_clean'].first()
    match_level['team_bat_2nd'] = match_level['match_id'].map(second_innings_team)
    decided3 = match_level.dropna(subset=['team_bat_2nd'])
    decided3 = decided3[decided3['won_by'] != 'Unknown'].copy()
    decided3['chase_won'] = decided3['team_bat_2nd'] == decided3['won_by']
    test3_p = binomtest(decided3['chase_won'].sum(), len(decided3), 0.5).pvalue

    return {
        'toss_winner_win_pct': round(n_toss_wins / n_total * 100, 1), 'test1_p': round(test1_p, 4),
        'decision_chi2': round(chi2, 3), 'test2_p': round(test2_p, 6),
        'chase_win_pct': round(decided3['chase_won'].mean() * 100, 1), 'test3_p': round(test3_p, 4),
    }

Appending to analysis.py


In [66]:
import importlib
import analysis
importlib.reload(analysis)
from analysis import compute_toss_chase_significance

results = compute_toss_chase_significance(df)
for k, v in results.items():
    print(k, ":", v)

toss_winner_win_pct : 51.6
test1_p : 0.2891
decision_chi2 : 9.123
test2_p : 0.002524
chase_win_pct : 54.3
test3_p : 0.0031


### Venue Advantage

In [69]:
%%writefile -a analysis.py

def compute_venue_toss_advantage(df, min_per_decision=10):
    """
    Phase 8: does the field-first advantage hold at every venue, or do
    some venues actually favor batting first?
    """
    df = df.copy()
    df['toss_winner_clean'] = df['toss_winner'].replace(TEAM_REBRAND_NAME)
    df['match_won_by_clean'] = df['match_won_by'].replace(TEAM_REBRAND_NAME)

    match_level = df.groupby('match_id').agg(
        venue=('venue_clean', 'first'), toss_winner=('toss_winner_clean', 'first'),
        toss_decision=('toss_decision', 'first'), won_by=('match_won_by_clean', 'first'),
    ).reset_index()
    decided = match_level[match_level['won_by'] != 'Unknown'].copy()
    decided['toss_winner_won'] = decided['toss_winner'] == decided['won_by']

    venue_field = decided[decided['toss_decision'] == 'field'].groupby('venue')['toss_winner_won'].agg(['mean', 'count'])
    venue_bat = decided[decided['toss_decision'] == 'bat'].groupby('venue')['toss_winner_won'].agg(['mean', 'count'])
    venue_field.columns, venue_bat.columns = ['field_win_pct', 'field_n'], ['bat_win_pct', 'bat_n']
    combined = venue_field.join(venue_bat, how='outer')
    combined['field_win_pct'] = (combined['field_win_pct'] * 100).round(1)
    combined['bat_win_pct'] = (combined['bat_win_pct'] * 100).round(1)

    qualified = combined[(combined['field_n'] >= min_per_decision) & (combined['bat_n'] >= min_per_decision)].copy()
    qualified['field_advantage'] = qualified['field_win_pct'] - qualified['bat_win_pct']
    return qualified.sort_values('field_advantage')

Appending to analysis.py


In [70]:
import importlib
import analysis
importlib.reload(analysis)
from analysis import compute_venue_toss_advantage

qualified = compute_venue_toss_advantage(df)
print(qualified)

                                              field_win_pct  field_n  \
venue                                                                  
MA Chidambaram Stadium                                 47.8     46.0   
Narendra Modi Stadium, Ahmedabad                       47.4     38.0   
Dubai International Cricket Stadium                    44.0     25.0   
Dr DY Patil Sports Academy                             55.6     27.0   
Arun Jaitley Stadium                                   54.0     63.0   
Wankhede Stadium                                       54.5    101.0   
Rajiv Gandhi International Stadium                     43.1     51.0   
Eden Gardens                                           55.6     72.0   
Punjab Cricket Association IS Bindra Stadium           52.3     44.0   
Sheikh Zayed Stadium                                   61.5     13.0   
Sawai Mansingh Stadium                                 67.5     40.0   

                                              bat_win_pct  bat_

## s death-overs scoring genuinely faster than powerplay?

In [21]:
%%writefile -a analysis.py

def compute_powerplay_death_ttest(df):
    """Phase 9, Test 3: paired t-test - death-overs vs powerplay run rate, innings by innings."""
    from scipy.stats import ttest_rel

    def phase_rr(sub):
        legal = (sub['valid_ball'] == 1).sum()
        return sub['runs_total'].sum() / (legal / 6) if legal > 0 else None

    pp_by_innings = df[df['over'] <= 5].groupby(['match_id', 'innings']).apply(phase_rr, include_groups=False)
    death_by_innings = df[df['over'] >= 15].groupby(['match_id', 'innings']).apply(phase_rr, include_groups=False)

    paired = pd.DataFrame({'pp_rr': pp_by_innings, 'death_rr': death_by_innings}).dropna()
    t_stat, p_val = ttest_rel(paired['death_rr'], paired['pp_rr'])
    mean_diff = round((paired['death_rr'] - paired['pp_rr']).mean(), 2)
    return round(t_stat, 3), p_val, mean_diff, len(paired)

Appending to analysis.py


In [22]:
import pandas as pd
df = pd.read_csv("data/processed/IPL_Dataset_Cleaned.csv", low_memory=False)
import importlib
import analysis
importlib.reload(analysis)
from analysis import compute_powerplay_death_ttest

t_stat, p_val, mean_diff, n = compute_powerplay_death_ttest(df)
print(f"t={t_stat}, p={p_val}, mean diff={mean_diff} runs/over, n={n} innings")

E:\neha work\Data Analytics\Projects\IPL sales analaysis\analysis.py:582: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pp_by_innings = df[df['over'] <= 5].groupby(['match_id', 'innings']).apply(phase_rr)
E:\neha work\Data Analytics\Projects\IPL sales analaysis\analysis.py:583: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  death_by_innings = df[df['over'] >= 15].groupby(['match_id', 'innings']).apply(phase_rr)


t=27.01, p=3.3572470322135823e-140, mean diff=2.05 runs/over, n=2356 innings


E:\neha work\Data Analytics\Projects\IPL sales analaysis\analysis.py:609: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pp_by_innings = df[df['over'] <= 5].groupby(['match_id', 'innings']).apply(phase_rr)
E:\neha work\Data Analytics\Projects\IPL sales analaysis\analysis.py:610: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  death_by_innings = df[df['over'] >= 15].groupby(['match_id', 'innings']).apply(phase_rr)


t=27.01, p=3.3572470322135823e-140, mean diff=2.05 runs/over, n=2356 innings
t=27.01, p=3.3572470322135823e-140, mean diff=2.05 runs/over, n=2356 innings


## Why Match Decisiveness Rate matters as its own KPI

In [23]:
%%writefile -a analysis.py

def compute_kpis(df):
    """Phase 10: headline KPI layer for the dashboard."""
    teams_raw = set(df['batting_team'].unique()) | set(df['bowling_team'].unique())
    players = set(df['batter'].unique()) | set(df['bowler'].unique()) | set(df['non_striker'].unique())
    decided_matches = df[df['win_outcome'].notna()]['match_id'].unique()
    regular = df[(df['innings'] <= 2) & (df['match_id'].isin(decided_matches))]
    innings_scores = regular.groupby(['match_id', 'innings'])['team_runs'].max()

    top_scorer = df.groupby('batter')['runs_batter'].sum().idxmax()
    top_scorer_runs = df.groupby('batter')['runs_batter'].sum().max()
    top_wkt_taker = df.groupby('bowler')['bowler_wicket'].sum().idxmax()
    top_wkt_taker_wkts = df.groupby('bowler')['bowler_wicket'].sum().max()

    return {
        'total_matches': df['match_id'].nunique(),
        'total_seasons': df['season_clean'].nunique(),
        'total_teams': 15,
        'total_players': len(players),
        'total_runs': int(df['runs_total'].sum()),
        'total_wickets': int(df['striker_out'].sum()),
        'average_score': round(innings_scores.mean(), 1),
        'highest_score': int(innings_scores.max()),
        'top_run_scorer': f"{top_scorer} ({top_scorer_runs})",
        'top_wicket_taker': f"{top_wkt_taker} ({top_wkt_taker_wkts})",
        'match_decisiveness_pct': round(len(decided_matches) / df['match_id'].nunique() * 100, 1),
    }



Appending to analysis.py


In [24]:
import importlib
import analysis
importlib.reload(analysis)
from analysis import compute_kpis

kpis = compute_kpis(df)
for k, v in kpis.items():
    print(k, ":", v)

E:\neha work\Data Analytics\Projects\IPL sales analaysis\analysis.py:582: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pp_by_innings = df[df['over'] <= 5].groupby(['match_id', 'innings']).apply(phase_rr)
E:\neha work\Data Analytics\Projects\IPL sales analaysis\analysis.py:583: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  death_by_innings = df[df['over'] >= 15].groupby(['match_id', 'innings']).apply(phase_rr)


t=27.01, p=3.3572470322135823e-140, mean diff=2.05 runs/over, n=2356 innings


E:\neha work\Data Analytics\Projects\IPL sales analaysis\analysis.py:609: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  pp_by_innings = df[df['over'] <= 5].groupby(['match_id', 'innings']).apply(phase_rr)
E:\neha work\Data Analytics\Projects\IPL sales analaysis\analysis.py:610: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  death_by_innings = df[df['over'] >= 15].groupby(['match_id', 'innings']).apply(phase_rr)


t=27.01, p=3.3572470322135823e-140, mean diff=2.05 runs/over, n=2356 innings
total_matches : 1243
total_seasons : 19
total_teams : 15
total_players : 807
total_runs : 401738
total_wickets : 14134
average_score : 162.2
highest_score : 287
top_run_scorer : V Kohli (9346)
top_wicket_taker : YS Chahal (233)
match_decisiveness_pct : 98.0
